# 02 — Factor Analysis

Explore Alpha158 features and custom A-share factors:
- Inspect Alpha158 dataset features
- Show custom factor expressions
- Single-factor IC analysis
- Factor cross-correlation heatmap

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 120

print(f"Project root: {PROJECT_ROOT}")

## 1. Alpha158 Dataset Features

In [ ]:
from big_a.factors.handler import create_alpha158_dataset, get_train_data

dataset = create_alpha158_dataset()
features_df = get_train_data(dataset, col_set="feature")

print(f"Alpha158 features shape: {features_df.shape}")
print(f"Feature columns (first 20): {list(features_df.columns[:20])}")
print(f"Total features: {len(features_df.columns)}")
print(f"\nSample data:")
features_df.iloc[:5, :8]

## 2. Custom A-Share Factor Expressions

In [ ]:
from big_a.factors.alpha_factors import CUSTOM_FACTORS, FACTOR_ALIASES

print("Custom factor expressions:")
print("=" * 60)
for alias, expr in FACTOR_ALIASES.items():
    print(f"  {alias:25s} = {expr}")

print(f"\nTotal custom factors: {len(CUSTOM_FACTORS)}")

## 3. Single-Factor IC Analysis

Compute the Information Coefficient (Pearson correlation between factor values and forward returns) for each custom factor.

In [ ]:
from qlib.data import D
from big_a.qlib_config import init_qlib
from big_a.backtest.evaluation import calc_ic, calc_rank_ic

init_qlib()

# Get a small set of instruments for quick analysis
instruments = list(D.instruments("csi300"))[:50]

# Forward return label
label_expr = ["Ref($close, -2) / Ref($close, -1) - 1"]
label_data = D.features(
    instruments,
    fields=label_expr,
    start_time="2024-01-01",
    end_time="2024-06-30",
)
label_data.columns = ["score"]

In [ ]:
ic_results = []

for alias, expr in FACTOR_ALIASES.items():
    try:
        factor_data = D.features(
            instruments,
            fields=[expr],
            start_time="2024-01-01",
            end_time="2024-06-30",
        )
        factor_data.columns = ["score"]

        # Align and compute IC
        ic = calc_ic(factor_data, label_data)
        rank_ic = calc_rank_ic(factor_data, label_data)

        ic_results.append({
            "factor": alias,
            "mean_ic": ic.mean() if len(ic) > 0 else np.nan,
            "mean_rank_ic": rank_ic.mean() if len(rank_ic) > 0 else np.nan,
            "ic_std": ic.std() if len(ic) > 0 else np.nan,
            "icir": ic.mean() / ic.std() if len(ic) > 1 and ic.std() > 0 else np.nan,
        })
    except Exception as e:
        print(f"  Skipping {alias}: {e}")

ic_df = pd.DataFrame(ic_results).set_index("factor")
print("\nSingle-Factor IC Summary:")
print(ic_df.to_string(float_format="%.4f"))

In [ ]:
# Visualize factor IC comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ic_df["mean_ic"].plot.barh(ax=ax, color="steelblue", alpha=0.8)
ax.set_title("Mean IC by Factor")
ax.set_xlabel("Mean IC")
ax.axvline(0, color="black", linewidth=0.5)

ax = axes[1]
ic_df["mean_rank_ic"].plot.barh(ax=ax, color="coral", alpha=0.8)
ax.set_title("Mean Rank IC by Factor")
ax.set_xlabel("Mean Rank IC")
ax.axvline(0, color="black", linewidth=0.5)

plt.tight_layout()
plt.show()

## 4. Factor Cross-Correlation Heatmap

In [ ]:
# Build a DataFrame with all factor values for correlation analysis
factor_frames = {}
for alias, expr in FACTOR_ALIASES.items():
    try:
        fd = D.features(
            instruments,
            fields=[expr],
            start_time="2024-01-01",
            end_time="2024-06-30",
        )
        fd.columns = [alias]
        factor_frames[alias] = fd[alias]
    except Exception:
        pass

combined = pd.DataFrame(factor_frames)
corr_matrix = combined.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr_matrix.columns)))
ax.set_yticks(range(len(corr_matrix.columns)))
ax.set_xticklabels(corr_matrix.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(corr_matrix.columns, fontsize=8)

for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center", va="center", fontsize=7,
                color="white" if abs(corr_matrix.iloc[i, j]) > 0.5 else "black")

plt.colorbar(im, ax=ax, label="Correlation")
ax.set_title("Factor Cross-Correlation Matrix")
plt.tight_layout()
plt.show()